# 🦀 Day 7: Mini-Project — Text Analyzer

Build a text analyzer that demonstrates proper ownership & borrowing.
Zero unnecessary clones!

---

## 📊 Step 1: Core Analysis Functions

In [ ]:
use std::collections::HashMap;

// All functions BORROW the text — no cloning needed!

fn word_count(text: &str) -> usize {
    text.split_whitespace().count()
}

fn char_count(text: &str) -> usize {
    text.chars().count()
}

fn sentence_count(text: &str) -> usize {
    text.split(|c: char| c == '.' || c == '!' || c == '?')
        .filter(|s| !s.trim().is_empty())
        .count()
}

fn average_word_length(text: &str) -> f64 {
    let words: Vec<&str> = text.split_whitespace().collect();
    if words.is_empty() { return 0.0; }
    let total: usize = words.iter().map(|w| w.trim_matches(|c: char| !c.is_alphanumeric()).len()).sum();
    total as f64 / words.len() as f64
}

let text = "Rust is a systems programming language. It ensures memory safety! How does it do that? Through ownership.";
println!("📊 Basic Stats:");
println!("  Words: {}", word_count(text));
println!("  Characters: {}", char_count(text));
println!("  Sentences: {}", sentence_count(text));
println!("  Avg word length: {:.1}", average_word_length(text));

## 📈 Step 2: Frequency Analysis

In [ ]:
// Returns borrowed slices — no allocations!
fn word_frequencies<'a>(text: &'a str) -> HashMap<&'a str, usize> {
    let mut freq = HashMap::new();
    for word in text.split_whitespace() {
        let clean = word.trim_matches(|c: char| !c.is_alphanumeric());
        if !clean.is_empty() {
            *freq.entry(clean).or_insert(0) += 1;
        }
    }
    freq
}

fn top_n_words<'a>(freq: &'a HashMap<&str, usize>, n: usize) -> Vec<(&'a &'a str, &'a usize)> {
    let mut sorted: Vec<_> = freq.iter().collect();
    sorted.sort_by(|a, b| b.1.cmp(a.1));
    sorted.into_iter().take(n).collect()
}

fn unique_word_count(freq: &HashMap<&str, usize>) -> usize {
    freq.len()
}

let text = "the quick brown fox jumps over the lazy dog the fox the dog";
let freq = word_frequencies(text);

println!("\n📈 Frequency Analysis:");
println!("  Unique words: {}", unique_word_count(&freq));
println!("  Top 5 words:");
for (word, count) in top_n_words(&freq, 5) {
    println!("    '{}' — {} times", word, count);
}

## 🔍 Step 3: Longest/Shortest

In [ ]:
// Returns references — zero-copy!
fn longest_word(text: &str) -> &str {
    text.split_whitespace()
        .max_by_key(|w| w.len())
        .unwrap_or("")
}

fn shortest_word(text: &str) -> &str {
    text.split_whitespace()
        .min_by_key(|w| w.len())
        .unwrap_or("")
}

fn longest_sentence(text: &str) -> &str {
    text.split(|c: char| c == '.' || c == '!' || c == '?')
        .map(|s| s.trim())
        .filter(|s| !s.is_empty())
        .max_by_key(|s| s.len())
        .unwrap_or("")
}

let text = "Rust is fast. Memory safety without garbage collection! Zero-cost abstractions make it efficient.";
println!("\n🔍 Length Analysis:");
println!("  Longest word: '{}'", longest_word(text));
println!("  Shortest word: '{}'", shortest_word(text));
println!("  Longest sentence: '{}'", longest_sentence(text));

## 🏗️ Step 4: Complete Analyzer

In [ ]:
struct TextAnalysis<'a> {
    text: &'a str,
    words: usize,
    chars: usize,
    sentences: usize,
    avg_word_len: f64,
    unique_words: usize,
}

impl<'a> TextAnalysis<'a> {
    fn analyze(text: &'a str) -> Self {
        let freq = word_frequencies(text);
        TextAnalysis {
            text,
            words: word_count(text),
            chars: char_count(text),
            sentences: sentence_count(text),
            avg_word_len: average_word_length(text),
            unique_words: unique_word_count(&freq),
        }
    }

    fn reading_time_seconds(&self) -> f64 {
        self.words as f64 / 200.0 * 60.0 // 200 WPM
    }

    fn vocabulary_richness(&self) -> f64 {
        if self.words == 0 { return 0.0; }
        self.unique_words as f64 / self.words as f64
    }

    fn report(&self) {
        println!("╔══════════════════════════════════════╗");
        println!("║       📊 TEXT ANALYSIS REPORT        ║");
        println!("╠══════════════════════════════════════╣");
        println!("║ Words:            {:>6}             ║", self.words);
        println!("║ Characters:       {:>6}             ║", self.chars);
        println!("║ Sentences:        {:>6}             ║", self.sentences);
        println!("║ Unique words:     {:>6}             ║", self.unique_words);
        println!("║ Avg word length:  {:>6.1}             ║", self.avg_word_len);
        println!("║ Reading time:     {:>5.0}s             ║", self.reading_time_seconds());
        println!("║ Vocab richness:   {:>5.0}%             ║", self.vocabulary_richness() * 100.0);
        println!("╚══════════════════════════════════════╝");
    }
}

let sample = "Rust is a multi-paradigm, general-purpose programming language. \
Rust emphasizes performance, type safety, and concurrency. \
Rust enforces memory safety without requiring a garbage collector. \
It achieves memory safety through a system of ownership with a set of rules. \
The Rust compiler checks these rules at compile time. \
If any of the rules are violated, the program will not compile. \
Rust is syntactically similar to C++, but can guarantee memory safety.";

let analysis = TextAnalysis::analyze(sample);
analysis.report();

// The text is NEVER cloned — everything is borrowed!
println!("\n✅ Zero clones — all analysis done through references!");

## 🏋️ Extension Challenges

In [ ]:
// Challenge 1: Add a method that returns the Flesch reading ease score
// Formula: 206.835 - 1.015 * (words/sentences) - 84.6 * (syllables/words)
// Approximate syllables by counting vowel groups

// YOUR CODE HERE


In [ ]:
// Challenge 2: Add a search function that returns all sentences
// containing a given keyword (returning &str slices)

// YOUR CODE HERE


## 🎯 Key Takeaways

1. **Zero-clone analysis**: all functions borrow text, no unnecessary copies
2. **Slice returns** (`&str`): functions return views into original data
3. **Lifetime annotations** (`'a`): used when structs hold references
4. **HashMap<&str, usize>**: keys are slices into original text
5. Ownership patterns in action: read with `&`, modify with `&mut`, consume with `self`

---
**Next Week:** Lifetimes & Generics! ⏳